# Data Center Vision-Stream Impact Classification Pipeline

Orchestrates tabular preprocessing, Earth Engine Sentinel-2 extraction, GCS upload, and optional Vertex AI training.

## Section 1: Dependencies & Environment Setup

In [ ]:
import os
import sys

import ee
import matplotlib.pyplot as plt
import torch
from IPython.display import display

from src.eda import (
    load_building_context,
    plot_geographic_tile_footprints,
    plot_satellite_samples,
    plot_tabular_eda,
    verify_tile_alignment,
)
from src.gcs import GCS_BUCKET, GCS_INPUT_PREFIX, upload_training_data
from src.preprocessing import run_tabular_preprocessing
from src.satellite import fetch_sentinel2_tensors, generate_mock_satellite_tensors

PROJECT_ID = "datacenter-summer-poc"
LOCATION = "us-central1"

ee.Initialize(project=PROJECT_ID)

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"GCP Project: {PROJECT_ID}, Region: {LOCATION}")

## Section 2: Tabular Preprocessing & Labels

In [ ]:
manifest_df = run_tabular_preprocessing("data/buildings.csv")
print(manifest_df["target_label"].value_counts().sort_index())

## Section 2b: Tabular EDA

Explore label balance, floor-area distributions, and geographic coverage before fetching imagery.

In [ ]:
context_df = load_building_context("data/parsed_manifest.csv")
display(context_df.head(10))

plot_tabular_eda(context_df)
plt.show()

# Sample records across all three impact tiers
sample_ids = (
    context_df.groupby("target_label", group_keys=False)
    .apply(lambda group: group.head(2))
    .sort_values("target_label")
)
display(
    sample_ids[
        ["OBJECTID", "BuildingName", "latitude", "longitude", "target_label", "MaxGFA"]
    ]
)

## Section 3: Satellite Image Pipeline (Earth Engine)

Set `USE_MOCK = True` for offline development without Earth Engine credentials. Use `limit=N` to extract a subset during dev.

In [ ]:
os.makedirs("data/image_tiles", exist_ok=True)
os.makedirs("data/raw_satellite", exist_ok=True)

USE_MOCK = False
DEV_LIMIT = 5  # set to None for full extraction (~202 tiles)

if USE_MOCK:
    tile_count = generate_mock_satellite_tensors(
        manifest_path="data/parsed_manifest.csv",
        output_dir="data/image_tiles",
        raw_output_dir="data/raw_satellite",
    )
else:
    tile_count = fetch_sentinel2_tensors(
        manifest_path="data/parsed_manifest.csv",
        output_dir="data/image_tiles",
        raw_output_dir="data/raw_satellite",
        project_id=PROJECT_ID,
        limit=DEV_LIMIT,
    )
print(f"Extracted {tile_count} model-ready tiles and raw satellite previews")

## Section 3b: Satellite Imagery EDA

Verify each building has aligned raw previews and processed tensors.

In [ ]:
alignment_report = verify_tile_alignment(
    manifest_path="data/parsed_manifest.csv",
    tile_dir="data/image_tiles",
    raw_dir="data/raw_satellite",
)
assert alignment_report["has_npy"].all(), "Some buildings are missing .npy tiles"
assert alignment_report["has_raw_rgb"].all(), (
    "Some buildings are missing raw RGB previews"
)
display(alignment_report.head(10))

## Section 3c: Satellite Tile Visualization

In [ ]:
qa_object_ids = (
    context_df.sort_values("MaxGFA", ascending=False)
    .groupby("target_label", group_keys=False)
    .head(1)["OBJECTID"]
    .astype(int)
    .tolist()
)
print("QA sample OBJECTIDs:", qa_object_ids)

plot_geographic_tile_footprints(context_df, sample_object_ids=qa_object_ids)
plt.show()

plot_satellite_samples(
    context_df,
    raw_dir="data/raw_satellite",
    tile_dir="data/image_tiles",
    sample_object_ids=qa_object_ids,
)
plt.show()

## Section 3d: GCS Upload

Sync manifest, tiles, and raw previews to the Vertex AI input prefix.

In [ ]:
gcs_uri = upload_training_data(
    local_data_dir="data",
    bucket=GCS_BUCKET,
    prefix=GCS_INPUT_PREFIX,
)
print(f"Training data available at {gcs_uri}")

## Section 4: Verify Production Modules

In [ ]:
from pathlib import Path

required_modules = [
    "src/model_def.py",
    "src/vertex_entrypoint.py",
    "src/gcs.py",
    "src/logging_config.py",
]
for module_path in required_modules:
    assert Path(module_path).exists(), f"Missing: {module_path}"
print("All production modules present in src/.")

## Section 5: Vertex AI Job Orchestration

Uncomment the cell below when GCP credentials and Vertex AI APIs are configured.

In [ ]:
# from google.cloud import aiplatform
#
# def deploy_vertex_training() -> None:
#     container = (
#         "us-docker.pkg.dev/vertex-ai/training/pytorch-gpu.2-0.py310:latest"
#     )
#     aiplatform.init(project=PROJECT_ID, location=LOCATION)
#     job = aiplatform.CustomTrainingJob(
#         display_name="datacenter-vision-train",
#         script_path="src/vertex_entrypoint.py",
#         container_uri=container,
#         requirements=["torch", "torchvision", "pandas", "numpy"],
#     )
#     job.run(
#         args=["--epochs", "10", "--batch-size", "8"],
#         replica_count=1,
#         machine_type="n1-standard-4",
#         accelerator_type="NVIDIA_TESLA_T4",
#         accelerator_count=1,
#         base_output_dir=f"gs://{GCS_BUCKET}/output/models/",
#     )
#
# deploy_vertex_training()